# Trening rdzenia GTO (Colab, POKER-69)

Runtime: **CPU**. GPU wyłączone — nie liczy artefaktu.
Gałąź: `grok/poker-53-aivat`. Profil `smoke` najpierw, `wta25` dopiero po zielonym dymie.
`TREE_ID=iso` (domyślnie) albo `call-v0` — call ma **osobny** OUT, nie nadpisuj dymu iso.
Sesja pada ~12 h. Bezpiecznik `--session-hours` (fuse po kompletnym cyklu/warstwie).
OUT na Drive. Artefaktu nie wrzucaj do publicznego repo (decyzja 30).
Tożsamość katalogu PROD: `colab_run.py identity --dir ...` — nie wołaj na OUT solvera.
Komórka 2 woła subprocess z listą argv — bez magii ! i interpolacji klamer.

In [ ]:
# 1. środowisko
import os, sys
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
REPO = "/content/Poker"
if not os.path.isdir(REPO):
    %cd /content
    !git clone --branch grok/poker-53-aivat https://github.com/mcz91/Poker.git
    !pip install -e "/content/Poker[train]"
else:
    !git -C /content/Poker pull --ff-only origin grok/poker-53-aivat
from google.colab import drive
drive.mount("/content/drive")
PROFILE = "smoke"  # potem: tdeep | wta25
TREE_ID = "iso"  # call-v0: osobny katalog OUT, nie nadpisuj dymu iso
os.environ["POKER_REPO"] = REPO
os.environ["POKER_PROFILE"] = PROFILE
os.environ["POKER_TREE_ID"] = TREE_ID
os.environ["POKER_TENSOR"] = f"{REPO}/tools/blueprint/control/tensor" if PROFILE == "smoke" else "/content/drive/MyDrive/poker-gto/PROD/tensor"
suffix = "" if TREE_ID == "iso" else f"-{TREE_ID}"
os.environ["POKER_OUT"] = f"/content/drive/MyDrive/poker-gto/{PROFILE}{suffix}"
os.environ["POKER_JOBS"] = str(os.cpu_count() or 2)
print({k: os.environ[k] for k in ("POKER_PROFILE", "POKER_TREE_ID", "POKER_TENSOR", "POKER_OUT", "POKER_JOBS")})

In [ ]:
# 2. solve — po restarcie odpal TĘ SAMĄ komórkę
import os, sys, subprocess
from pathlib import Path
script = str(Path(os.environ["POKER_REPO"]) / "tools" / "blueprint" / "colab_run.py")
out = os.environ["POKER_OUT"]
hours = "0.25" if os.environ["POKER_PROFILE"] == "smoke" else "9"
cmd = [
    sys.executable, script, "solve",
    "--profile", os.environ["POKER_PROFILE"],
    "--tensor", os.environ["POKER_TENSOR"],
    "--out", out,
    "--session-hours", hours,
    "--jobs", os.environ["POKER_JOBS"],
    "--tree-id", os.environ.get("POKER_TREE_ID", "iso"),
]
if not (Path(out) / "solve_manifest.json").exists():
    cmd.append("--allow-fresh")
print(" ".join(cmd))
subprocess.check_call(cmd)
subprocess.check_call([sys.executable, script, "status", "--out", out])

In [ ]:
# 3. pack tylko gdy status=done
import os, sys, subprocess
from pathlib import Path
script = str(Path(os.environ["POKER_REPO"]) / "tools" / "blueprint" / "colab_run.py")
out = os.environ["POKER_OUT"]
subprocess.check_call([
    sys.executable, script, "pack",
    "--run", out,
    "--bpk", str(Path(out) / "blueprint_v2.bpk"),
])